## 1. Install (run once)

Runtime > Change runtime type > choose an available GPU first.
Keep Colab's preinstalled matched torch/torchvision pair.
After this cell finishes, choose **Runtime > Restart session**.
Then continue from section 2; do not repeatedly run the install cell.


In [ ]:
import subprocess, sys
packages = [
    "transformers==4.57.1", "opencv-python-headless==4.12.0.88",
    "numpy>=2.0,<2.3", "Pillow>=10,<14", "imageio-ffmpeg==0.6.0",
    "safetensors>=0.4.3",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Install complete. Restart session now, then continue from section 2.")


Install complete. Restart session now, then continue from section 2.


## 2. Environment check (after restart)

If torchvision import fails, start a fresh GPU runtime and retain its matched
torch/torchvision pair. CPU is allowed but can be very slow. First model run needs
internet to download public checkpoints.


In [ ]:
import sys, json, subprocess, shutil, csv
from pathlib import Path
import torch, torchvision, transformers, cv2, numpy as np
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation
print("Python:", sys.version)
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__, "OpenCV:", cv2.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("CPU selected. Prefer an available GPU for the actual ML run.")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
torch: 2.14.1+cu130 torchvision: 0.29.1+cu130
transformers: 4.57.1 OpenCV: 4.12.0
CUDA: True


## 3. Write the corrected detector

This cell embeds the complete V4 implementation. Existing boards and clean wall
interiors are detected; people, doors/windows/cars and detected product boxes are
used only as exclusion masks. They are never displayed as ad slots.


In [ ]:
%%writefile ad_slot_detector_v4.py
"""Phase 3 V4: detected billboards/walls + verified temporal suggestions.

Uses pretrained Grounding DINO. Existing ad boards are included by default.
Read FIXED_README_HINGLISH.md. Candidate detection and 4s eligibility are separate.
Usage: python ad_slot_detector_v4.py --video input.mp4 --out-dir results/video1
Only clean original videos should be used for scientific evaluation.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import inspect
import json
import math
import platform
import shutil
import subprocess
import time
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path

import cv2
import numpy as np


@dataclass
class Config:
    max_slots: int = 3                 # TOTAL recommendations per video
    min_duration: float = 4.0
    target_duration: float = 5.0
    detect_hz: float = 1.0
    min_conf: float = 0.28
    text_threshold: float = 0.25
    min_surface_fraction: float = 0.88
    min_area: float = 0.004
    max_area: float = 0.55
    min_width: float = 0.075
    min_height: float = 0.035
    min_rank: float = 0.46
    match_iou: float = 0.40
    max_edge_density: float = 0.08
    max_texture_std: float = 25.0
    allow_existing: bool = True
    model_id: str = 'IDEA-Research/grounding-dino-tiny'


def area(box):
    return max(0., float(box[2] - box[0])) * max(0., float(box[3] - box[1]))


def intersection(a, b):
    return max(0., min(a[2], b[2]) - max(a[0], b[0])) * max(0., min(a[3], b[3]) - max(a[1], b[1]))


def iou(a, b):
    inter = intersection(a, b)
    return inter / max(1., area(a) + area(b) - inter)


def same_region(a, b):
    return iou(a, b) > 0.40 or intersection(a, b) / max(1., min(area(a), area(b))) > 0.75


def clipped(box, width, height):
    x1, y1, x2, y2 = map(float, box)
    return [max(0., min(width, x1)), max(0., min(height, y1)),
            max(0., min(width, x2)), max(0., min(height, y2))]


def ints(box):
    return [int(round(v)) for v in box]


def largest_rectangle(mask):
    """Largest axis-aligned ALL-TRUE rectangle. Returns exclusive x2/y2."""
    height, width = mask.shape
    histogram = np.zeros(width, dtype=int)
    best, best_area = None, 0
    for y in range(height):
        histogram = np.where(mask[y], histogram + 1, 0)
        stack = []
        for x in range(width + 1):
            current = int(histogram[x]) if x < width else 0
            left = x
            while stack and stack[-1][1] > current:
                start, bar = stack.pop()
                candidate_area = bar * (x - start)
                if candidate_area > best_area:
                    best_area, best = candidate_area, [start, y + 1 - bar, x, y + 1]
                left = start
            if current and (not stack or stack[-1][1] < current):
                stack.append((left, current))
    return best


def crop_rectangle(mask, proposal):
    """Carve a usable interior from a model proposal; never infer an ad from OCR."""
    h, w = mask.shape
    x1, y1, x2, y2 = ints(clipped(proposal, w, h))
    if x2 <= x1 or y2 <= y1:
        return None
    roi = mask[y1:y2, x1:x2].astype(np.uint8)
    scale = min(1., 160 / max(roi.shape))
    mw, mh = max(1, int(roi.shape[1] * scale)), max(1, int(roi.shape[0] * scale))
    # Area resize + threshold avoids turning small unsafe holes into valid pixels.
    small = cv2.resize(roi.astype(np.float32), (mw, mh), interpolation=cv2.INTER_AREA) > 0.995
    rect = largest_rectangle(small)
    if rect is None:
        return None
    a, b, c, d = rect
    result = [x1 + math.ceil(a * roi.shape[1] / mw), y1 + math.ceil(b * roi.shape[0] / mh),
              x1 + math.floor(c * roi.shape[1] / mw), y1 + math.floor(d * roi.shape[0] / mh)]
    # An ad typically needs a landscape interior. Crop a very wide/tall result.
    a, b, c, d = result
    bw, bh = c - a, d - b
    if bw < 1 or bh < 1:
        return None
    if bw / bh < 1.25:
        nh = bw / 1.25
        b, d = (b + d - nh) / 2, (b + d + nh) / 2
    elif bw / bh > 4.0:
        nw = bh * 4.0
        a, c = (a + c - nw) / 2, (a + c + nw) / 2
    return ints([a, b, c, d])


def geometry_ok(box, width, height, cfg, kind='existing_board'):
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    ratio = area(box) / (width * height)
    return (bw >= cfg.min_width * width and bh >= cfg.min_height * height
            and cfg.min_area <= ratio <= cfg.max_area
            and 0.85 <= bw / max(1., bh) <= 7.0
            and x1 >= width * 0.003 and y1 >= height * 0.003
            and x2 <= width * 0.997 and y2 <= height * 0.98
            and (kind != 'blank_surface' or ratio >= .008))


def appearance(frame, box):
    x1, y1, x2, y2 = ints(box)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return {'edge': 1., 'std': 255., 'clipped': 1.,'residual_std':255.,'residual_outliers':1.}
    gray = cv2.cvtColor(cv2.resize(crop, (160, 90)), cv2.COLOR_BGR2GRAY)
    # Remove a smooth illumination gradient before measuring wall irregularity.
    # Low-contrast writing/plaster can have few Canny edges and still be unsuitable.
    yy,xx=np.mgrid[:90,:160]
    design=np.column_stack((np.ones(gray.size),xx.ravel(),yy.ravel()))
    residual=gray.ravel()-design@np.linalg.lstsq(design,gray.ravel(),rcond=None)[0]
    return {'edge': float(np.mean(cv2.Canny(gray, 70, 150) > 0)),
            'std': float(gray.std()),
            'residual_std':float(residual.std()),
            'residual_outliers':float(np.mean(np.abs(residual)>12)),
            'clipped': float(np.mean((gray < 8) | (gray > 247)))}


def assess(frame, box, safe_mask, kind, confidence, cfg, protected_mask=None):
    h, w = frame.shape[:2]
    if not geometry_ok(box, w, h, cfg, kind):
        return None, 'geometry'
    x1, y1, x2, y2 = ints(box)
    if protected_mask is not None and float(protected_mask[y1:y2,x1:x2].mean()) > .015:
        return None, 'foreground_overlap'
    supported = float(safe_mask[y1:y2, x1:x2].mean())
    if supported < cfg.min_surface_fraction:
        return None, 'unsafe_or_uncertain_surface'
    q = appearance(frame, box)
    if kind == 'blank_surface' and (q['edge'] > cfg.max_edge_density or q['std'] > cfg.max_texture_std
                                    or q['residual_std']>10 or q['residual_outliers']>.12):
        return None, 'clutter_or_text'
    if q['clipped'] > 0.80:
        return None, 'extreme_dark_or_bright'
    cleanliness = 1. - min(1., q['edge'] / cfg.max_edge_density)
    size_score = min(1., area(box) / (w * h * 0.12))
    # A deterministic research ranking index, NOT a calibrated probability.
    score = (0.30 * confidence + 0.25 * supported + 0.20 * size_score
             + 0.15 * (cleanliness if kind == 'blank_surface' else 0.8) + 0.10)
    if score < cfg.min_rank:
        return None, 'low_rank'
    return dict(box=ints(box), kind=kind, confidence=float(confidence),
                score=float(score), surface_fraction=supported, appearance=q), 'accepted'


class Models:
    """One Grounding DINO pass for physical surfaces AND foreground protection.

    Existing boards are eligible by default. Generic semantic segmentation is
    not used as a compulsory gate for a zero-shot billboard detection.
    """
    PROMPT = 'a billboard. a signboard. a wall. a person. a window. a door. a car.'

    def __init__(self, cfg, device=None, cache_path=None):
        self.cfg = cfg
        self.requested_device = device
        self.device = device
        self.cache_path = Path(cache_path) if cache_path else None
        self.cache = {}
        self.product_checked = set()
        self.last_raw = []
        self.frame_index = 0
        self.detector = None
        self.proc = None
        self.inference_seconds = 0.
        if self.cache_path and self.cache_path.exists():
            data = json.loads(self.cache_path.read_text())
            if data.get('prompt') == self.PROMPT and data.get('model_id') == cfg.model_id:
                self.cache = data.get('frames', {})
                self.product_checked = set(data.get('product_checked', []))

    def load(self):
        import torch
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        torch.set_num_threads(min(4, __import__('os').cpu_count() or 4))
        self.torch = torch
        self.device = self.requested_device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.proc = AutoProcessor.from_pretrained(self.cfg.model_id)
        self.detector = AutoModelForZeroShotObjectDetection.from_pretrained(
            self.cfg.model_id, disable_custom_kernels=True).to(self.device).eval()
        print('Loaded Grounding DINO on', self.device, flush=True)

    def infer_prompt(self, frame, prompt):
        if self.detector is None:
            self.load()
        inputs = self.proc(images=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), text=prompt,
                           size={'shortest_edge': 640, 'longest_edge': 1024},
                           return_tensors='pt').to(self.device)
        started = time.time()
        with self.torch.inference_mode():
            outputs = self.detector(**inputs)
        self.inference_seconds += time.time() - started
        fn = self.proc.post_process_grounded_object_detection
        kwargs = dict(input_ids=inputs['input_ids'], text_threshold=.22, target_sizes=[frame.shape[:2]])
        kwargs['threshold' if 'threshold' in inspect.signature(fn).parameters else 'box_threshold'] = .20
        result = fn(outputs, **kwargs)[0]
        labels = result.get('text_labels', result.get('labels'))
        raw = []
        for box, score, label in zip(result['boxes'], result['scores'], labels):
            if not isinstance(label, str):
                continue
            raw.append({'box': box.detach().cpu().tolist(), 'confidence': float(score.detach().cpu()),
                        'label': label.lower()})
        return raw

    def proposals(self, frame):
        key = str(self.frame_index)
        cached = key in self.cache
        raw = self.cache[key] if cached else self.infer_prompt(frame,self.PROMPT)
        # A small board-like logo near the floor may actually be product packaging.
        # Ask the detector about packaging instead of accepting its brand text as
        # proof of an advertising surface. No product outlines are ever rendered.
        h,w = frame.shape[:2]
        needs_check = any(self.kind(p['label'])=='existing_board' and p['confidence']>=self.cfg.min_conf
                          and (p['box'][1]+p['box'][3])/(2*h)>.58
                          and area(p['box'])/(h*w)<.12 for p in raw)
        if needs_check and key not in self.product_checked:
            raw = raw + self.infer_prompt(frame,'a product box. a cardboard box. a carton.')
            self.product_checked.add(key)
            cached = False
        self.cache[key] = raw
        self.last_raw = raw
        if self.device is None:
            self.device = 'cached_model_detections'
        if self.cache_path:
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)
            self.cache_path.write_text(json.dumps({'prompt': self.PROMPT, 'model_id': self.cfg.model_id,
                                                   'frames': self.cache,
                                                   'product_checked': sorted(self.product_checked)}), encoding='utf-8')
        if not cached:
            print(f'Model frame {self.frame_index}: {len(raw)} raw detections', flush=True)
        return raw

    @staticmethod
    def kind(label):
        # Foreground words take priority if token grounding joins two phrases.
        if any(word in label for word in ('person', 'window', 'door', 'car', 'product', 'box', 'carton')):
            return 'protected'
        if 'billboard' in label or 'signboard' in label:
            return 'existing_board'
        if 'wall' in label:
            return 'blank_surface'
        return 'ignore'

    def masks(self, frame):
        raw = self.proposals(frame)
        h, w = frame.shape[:2]
        protected = np.zeros((h, w), np.uint8)
        wall_support = np.zeros((h, w), np.uint8)
        board_support = np.zeros((h, w), np.uint8)
        self.protected_objects = []
        for p in raw:
            kind = self.kind(p['label'])
            box = clipped(p['box'], w, h)
            x1,y1,x2,y2 = ints(box)
            if kind == 'protected' and p['confidence'] >= .25:
                # Bounding boxes, not outlines, are used only as exclusion masks.
                if area(box)/(h*w) >= .001:
                    margin = max(2, round(w*.003))
                    protected[max(0,y1-margin):min(h,y2+margin),max(0,x1-margin):min(w,x2+margin)] = 1
                    self.protected_objects.append(p)
            elif kind in ('existing_board','blank_surface'):
                threshold = self.cfg.min_conf if kind=='existing_board' else .22
                if p['confidence'] < threshold:
                    continue
                # Mild expansion absorbs detector jitter when an independently
                # tracked, smaller interior is checked at the next model sample.
                mx, my = .04*(x2-x1), .04*(y2-y1)
                a,b,c,d = ints(clipped([x1-mx,y1-my,x2+mx,y2+my],w,h))
                target = board_support if kind=='existing_board' else wall_support
                target[b:d,a:c] = 1
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edge = (cv2.Canny(cv2.GaussianBlur(gray,(3,3),0),60,140)>0).astype(np.float32)
        local_density = cv2.boxFilter(edge, -1, (21,21))
        blank = (wall_support>0) & (protected==0) & (local_density < .075)
        blank = cv2.erode(blank.astype(np.uint8),np.ones((5,5),np.uint8)).astype(bool)
        return {'blank_surface':blank, 'existing_board':(board_support>0)&(protected==0),
                'foreground':protected.astype(bool)}

    def detect(self, frame, masks, rejects):
        h,w = frame.shape[:2]
        candidates = []
        for p in sorted(self.last_raw,key=lambda p:p['confidence'],reverse=True):
            kind = self.kind(p['label'])
            if kind not in ('blank_surface','existing_board'):
                continue
            if kind=='existing_board' and not self.cfg.allow_existing:
                rejects['existing_board_disabled'] += 1
                continue
            if p['confidence'] < (self.cfg.min_conf if kind=='existing_board' else .22):
                rejects['low_model_confidence'] += 1
                continue
            box = clipped(p['box'],w,h)
            if kind=='blank_surface':
                x1,y1,x2,y2=box
                margin_x,margin_y=max(4,round(w*.012)),max(4,math.ceil(h*.025))
                box=[max(margin_x,x1),max(margin_y,y1),min(w-margin_x,x2),min(h-margin_y,y2)]
                box = crop_rectangle(masks[kind],box)
                if box is None:
                    rejects['no_clean_wall_interior'] += 1
                    continue
            else:
                x1,y1,x2,y2=box
                # Inset only a little: preserve the visible billboard face.
                box=ints([x1+.015*(x2-x1),y1+.015*(y2-y1),x2-.015*(x2-x1),y2-.015*(y2-y1)])
            item,reason=assess(frame,box,masks[kind],kind,p['confidence'],self.cfg,masks['foreground'])
            if item is None:
                rejects[reason] += 1
            elif all(not same_region(item['box'],other['box']) for other in candidates):
                item['label']=p['label']
                candidates.append(item)
        return sorted(candidates,key=lambda item:item['score'],reverse=True)[:6]



def scene_cut(previous, current):
    if previous is None:
        return False
    a = cv2.resize(previous, (160, 90))
    b = cv2.resize(current, (160, 90))
    ha = cv2.calcHist([cv2.cvtColor(a, cv2.COLOR_BGR2HSV)], [0, 1], None, [24, 24], [0, 180, 0, 256])
    hb = cv2.calcHist([cv2.cvtColor(b, cv2.COLOR_BGR2HSV)], [0, 1], None, [24, 24], [0, 180, 0, 256])
    cv2.normalize(ha, ha, alpha=1., norm_type=cv2.NORM_L1)
    cv2.normalize(hb, hb, alpha=1., norm_type=cv2.NORM_L1)
    histogram_change = cv2.compareHist(ha, hb, cv2.HISTCMP_BHATTACHARYYA)
    gray_change = float(np.mean(cv2.absdiff(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)))) / 255
    # The extra pixel threshold also catches cuts between similarly colored rooms.
    # Fast camera pans can deliberately cause conservative splits.
    return bool((histogram_change > .60 and gray_change > .10) or gray_change > .16)


def track_step(previous_gray, current_gray, box):
    """Local LK flow + forward/backward check + RANSAC; fail rather than drift."""
    h, w = current_gray.shape
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    pad = .15
    a, b, c, d = ints(clipped([x1 - pad * bw, y1 - pad * bh, x2 + pad * bw, y2 + pad * bh], w, h))
    region = np.zeros((h, w), np.uint8)
    region[b:d, a:c] = 255
    points = cv2.goodFeaturesToTrack(previous_gray, maxCorners=60, qualityLevel=.01,
                                    minDistance=6, mask=region, blockSize=5)
    if points is not None and len(points) >= 6:
        nxt, status, _ = cv2.calcOpticalFlowPyrLK(previous_gray, current_gray, points, None, winSize=(21, 21), maxLevel=3)
        if nxt is not None:
            back, back_status, _ = cv2.calcOpticalFlowPyrLK(current_gray, previous_gray, nxt, None, winSize=(21, 21), maxLevel=3)
            if back is not None:
                valid = (status.ravel() == 1) & (back_status.ravel() == 1) & (np.linalg.norm(points - back, axis=2).ravel() < 1.2)
                p, q = points[valid], nxt[valid]
                if len(p) >= 6:
                    matrix, inliers = cv2.estimateAffinePartial2D(p, q, method=cv2.RANSAC, ransacReprojThreshold=2.0)
                    if matrix is not None and inliers is not None and inliers.mean() >= .65:
                        angle = abs(math.degrees(math.atan2(matrix[1, 0], matrix[0, 0])))
                        corners = np.float32([[x1, y1], [x2, y1], [x2, y2], [x1, y2]]).reshape(-1, 1, 2)
                        moved = cv2.transform(corners, matrix).reshape(-1, 2)
                        new_box = [float(moved[:, 0].min()), float(moved[:, 1].min()),
                                   float(moved[:, 0].max()), float(moved[:, 1].max())]
                        scale = area(new_box) / max(1., area(box))
                        displacement = math.hypot(new_box[0] - x1, new_box[1] - y1)
                        if .85 < scale < 1.18 and angle < 3.0 and displacement < .08 * w:
                            if new_box[0] >= 0 and new_box[1] >= 0 and new_box[2] <= w and new_box[3] <= h:
                                # Check appearance after affine compensation to catch occluders.
                                warped = cv2.warpAffine(previous_gray, matrix, (w, h))
                                aa, bb, cc, dd = ints(new_box)
                                diff = cv2.absdiff(warped[bb:dd, aa:cc], current_gray[bb:dd, aa:cc])
                                if diff.size and np.mean(diff > 35) < .08 and diff.mean() < 12:
                                    return new_box, float(inliers.mean())
    # A truly stationary plain wall may have no feature points. Require pixel evidence.
    before, after = previous_gray[b:d, a:c], current_gray[b:d, a:c]
    if before.size:
        difference = cv2.absdiff(before, after)
        if difference.mean() < 3.0 and np.mean(difference > 20) < .015:
            return list(map(float, box)), .85
    return None, 0.


@dataclass
class Track:
    track_id: int
    scene_id: int
    kind: str
    label: str
    confidence: float
    box: list
    rows: list = field(default_factory=list)
    last_verified: int = -1
    rank: float = 0.


def window_options(tracks, fps, cfg):
    """Consecutive observed/tracked frames with periodic detector confirmation.

    Flow-verified frames up to the next scheduled detector sample count toward
    visibility. No missing, failed, frozen or padded frames are manufactured.
    """
    minimum = math.ceil(cfg.min_duration * fps - 1e-6)
    target = math.floor(cfg.target_duration * fps + 1e-6)
    options = []
    for tr in tracks:
        max_tracked_tail = max(0, math.ceil(fps/cfg.detect_hz)-1)
        rows = [r for r in tr.rows if r[0] <= tr.last_verified + max_tracked_tail]
        if len(rows) < minimum:
            continue
        # Defensive split: never bridge missing frames even if a caller passes them.
        segments, segment = [], []
        for row in rows:
            if segment and row[0] != segment[-1][0] + 1:
                segments.append(segment)
                segment = []
            segment.append(row)
        if segment:
            segments.append(segment)
        for rows in segments:
            if len(rows) < minimum:
                continue
            # Compare 4s through 5s durations, prefer 5s when scores are close.
            lengths = sorted(set([minimum, min(target, len(rows))]))
            for length in lengths:
                values = np.array([r[2] * (.85 + .15 * r[3]) for r in rows])
                means = np.convolve(values, np.ones(length) / length, mode='valid')
                # Include alternatives so a high-ranked interval does not block all others.
                for start in range(0, len(means), max(1, round(fps / 2))):
                    selected = rows[start:start + length]
                    rank = float(means[start]) + .02 * (length / max(1, target))
                    if rank >= cfg.min_rank:
                        options.append(dict(track_id=tr.track_id, scene_id=tr.scene_id, kind=tr.kind,
                                            label=tr.label, start_frame=selected[0][0],
                                            end_frame_exclusive=selected[-1][0] + 1, rank=min(1., rank),
                                            rows=selected, min_frame_score=float(min(r[2] for r in selected))))
    return sorted(options, key=lambda o: o['rank'], reverse=True)


def select_windows(options, cfg):
    """At most 3 TOTAL; one per track; no overlapping time intervals."""
    selected = []
    for option in options:
        if any(o['track_id'] == option['track_id'] for o in selected):
            continue
        if any(option['start_frame'] < o['end_frame_exclusive']
               and o['start_frame'] < option['end_frame_exclusive'] for o in selected):
            continue
        selected.append(option)
        if len(selected) == cfg.max_slots:
            break
    for rank, option in enumerate(selected, 1):
        option['rank_order'] = rank
    selected.sort(key=lambda o: o['start_frame'])
    for index, option in enumerate(selected, 1):
        option['slot_id'] = index
    return selected


def timestamp(seconds):
    millis = round(seconds * 1000)
    return f'{millis // 3600000:02d}:{millis // 60000 % 60:02d}:{millis // 1000 % 60:02d}.{millis % 1000:03d}'


def ffmpeg_path():
    path = shutil.which('ffmpeg')
    if path:
        return path
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError as exc:
        raise RuntimeError('Install FFmpeg or imageio-ffmpeg; see requirements-phase3.txt') from exc


def run_ffmpeg(command, log):
    with Path(log).open('w', encoding='utf-8') as f:
        result = subprocess.run(command, stdout=f, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'FFmpeg failed. Inspect {log}')


def normalize_video(source, destination, max_width):
    # Normalize VFR input to 30fps BEFORE computing timestamps and preserve audio.
    vf = f"scale=w='min({max_width},iw)':h=-2,scale=trunc(iw/2)*2:trunc(ih/2)*2,fps=30"
    run_ffmpeg([ffmpeg_path(), '-hide_banner', '-y', '-i', str(source), '-map', '0:v:0', '-map', '0:a:0?',
                '-vf', vf, '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-pix_fmt', 'yuv420p',
                '-c:a', 'aac', '-movflags', '+faststart', str(destination)], destination.with_suffix('.log'))


def analyze(video, models, cfg):
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open {video}')
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        raise RuntimeError('Invalid FPS; video normalization failed.')
    working_width, working_height = int(cap.get(3)), int(cap.get(4))
    step = max(1, round(fps / cfg.detect_hz))
    active, finished, rejects = [], [], Counter()
    previous, previous_gray = None, None
    frame_idx, next_id, scene_id, cuts = 0, 1, 0, []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        is_cut = scene_cut(previous, frame)
        if is_cut:
            finished.extend(active)
            active = []
            scene_id += 1
            cuts.append(frame_idx)
        if previous_gray is not None and not is_cut:
            survivors = []
            for tr in active:
                box, quality = track_step(previous_gray, gray, tr.box)
                if box is None:
                    rejects['tracking_failure'] += 1
                    finished.append(tr)
                else:
                    tr.box = box
                    tr.rows.append([frame_idx, ints(box), tr.rank, quality])
                    survivors.append(tr)
            active = survivors
        if frame_idx % step == 0 or is_cut:
            models.frame_index = frame_idx
            masks = models.masks(frame)
            # Verify each existing track at ITS tracked location, avoiding box jumps.
            survivors = []
            for tr in active:
                item, reason = assess(frame, tr.box, masks[tr.kind], tr.kind, tr.confidence, cfg,masks.get('foreground'))
                if item is None:
                    rejects['recheck_' + reason] += 1
                    if tr.rows and tr.rows[-1][0] == frame_idx:
                        tr.rows.pop()  # never show the current explicitly rejected frame
                    finished.append(tr)
                else:
                    tr.last_verified, tr.rank = frame_idx, item['score']
                    tr.rows[-1][2] = tr.rank
                    survivors.append(tr)
            active = survivors
            candidates = models.detect(frame, masks, rejects)
            # DINO does not have to redetect a stationary wall every time: semantic
            # checks + pixel/flow evidence must still succeed for every active track.
            for item in candidates:
                if any(iou(item['box'], tr.box) >= cfg.match_iou or same_region(item['box'], tr.box) for tr in active):
                    continue
                tr = Track(next_id, scene_id, item['kind'], item['label'], item['confidence'], item['box'],
                           rows=[[frame_idx, item['box'], item['score'], 1.]],
                           last_verified=frame_idx, rank=item['score'])
                active.append(tr)
                next_id += 1
            if frame_idx % max(step, round(fps * 5)) == 0:
                print(f'Analyzed {frame_idx / fps:.1f}s | active tracks={len(active)} | finished={len(finished)}', flush=True)
        previous, previous_gray = frame, gray
        frame_idx += 1
    cap.release()
    finished.extend(active)
    return finished, dict(fps=fps, processed_frames=frame_idx, duration_sec=frame_idx / fps,
                          coordinate_space='normalized_input_pixels', working_width=working_width,
                          working_height=working_height,
                          detect_every_frames=step, effective_detect_hz=fps / step,
                          scene_cut_frames=cuts, rejected_counts=dict(rejects))


def render(video, destination, selected, fps, preview=False):
    cap = cv2.VideoCapture(str(video))
    width, height = int(cap.get(3)), int(cap.get(4))
    silent = destination.parent / '_annotated_silent.mp4'
    writer = cv2.VideoWriter(str(silent), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError('OpenCV video writer failed.')
    frame_map = {}
    for slot in selected:
        for row in slot['rows']:
            frame_map.setdefault(row[0], []).append((slot, row[1]))
    index = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        for slot, box in sorted(frame_map.get(index, []), key=lambda x:x[0]['rank'], reverse=True)[:3]:
            x1, y1, x2, y2 = ints(box)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (50, 230, 60), 2)
            label = (f"DETECTED {'BOARD' if slot['kind']=='existing_board' else 'WALL SPACE'} | ID {slot['slot_id']}"
                     if preview else f"SLOT {slot['slot_id']} | {timestamp(slot['start_frame']/fps)} - {timestamp(slot['end_frame_exclusive']/fps)}")
            font_scale = max(.35, min(.6, width / 1900))
            text_w, text_h = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 1)[0]
            tx, ty = max(4, min(x1, width - text_w - 6)), max(text_h + 7, y1 - 8)
            cv2.rectangle(frame, (tx-2, ty-text_h-4), (tx+text_w+3, ty+4), (20, 25, 20), -1)
            cv2.putText(frame, label, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (90, 255, 100), 1, cv2.LINE_AA)
            if index == slot['start_frame'] and not preview:
                cv2.imwrite(str(destination.parent / f"slot_{slot['slot_id']}_preview.jpg"), frame)
        writer.write(frame)
        index += 1
    cap.release()
    writer.release()
    run_ffmpeg([ffmpeg_path(), '-hide_banner', '-y', '-i', str(silent), '-i', str(video),
                '-map', '0:v:0', '-map', '1:a:0?', '-c:v', 'libx264', '-preset', 'veryfast',
                '-crf', '20', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-movflags', '+faststart',
                str(destination)], destination.with_suffix('.log'))
    if not destination.exists() or destination.stat().st_size == 0:
        raise RuntimeError('Rendered output is empty.')
    silent.unlink(missing_ok=True)
    return index


def candidate_report(tracks, out_dir, fps, cfg):
    previews, summaries = [], []
    for track in tracks:
        rows = track.rows
        if len(rows) < max(2, round(.4*fps)):
            continue
        duration = len(rows)/fps
        verified_duration = sum(r[0] <= track.last_verified for r in rows)/fps
        cid = len(previews)+1
        previews.append(dict(slot_id=cid, track_id=track.track_id, kind=track.kind,
                             rows=rows, rank=float(np.mean([r[2] for r in rows])),
                             start_frame=rows[0][0], end_frame_exclusive=rows[-1][0]+1))
        summaries.append(dict(candidate_id=cid,track_id=track.track_id,kind=track.kind,
                              start_sec=rows[0][0]/fps,end_sec=(rows[-1][0]+1)/fps,
                              continuous_duration_sec=round(duration,3),
                              detector_confirmed_span_sec=round(verified_duration,3),
                              meets_4s_rule=duration>=cfg.min_duration,
                              reason='eligible_for_ranking' if duration>=cfg.min_duration else 'detected_but_visible_less_than_4s'))
    fields=['candidate_id','track_id','kind','start_sec','end_sec','continuous_duration_sec','detector_confirmed_span_sec','meets_4s_rule','reason']
    with (out_dir/'candidate_summary.csv').open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=fields)
        writer.writeheader();writer.writerows(summaries)
    return previews,summaries


def export_results(source, out_dir, cfg, selected, meta, started, models):
    fps = meta['fps']
    slots, coordinates = [], []
    for slot in selected:
        rows = slot['rows']
        first = rows[0][1]
        slots.append(dict(slot_id=slot['slot_id'], rank_order=slot['rank_order'],
                          track_id=slot['track_id'], scene_id=slot['scene_id'], kind=slot['kind'],
                          start_sec=slot['start_frame']/fps, end_sec=slot['end_frame_exclusive']/fps,
                          start_time=timestamp(slot['start_frame']/fps),
                          end_time=timestamp(slot['end_frame_exclusive']/fps),
                          duration_sec=len(rows)/fps, start_frame=slot['start_frame'],
                          end_frame_exclusive=slot['end_frame_exclusive'],
                          ranking_score=round(slot['rank'], 4), min_frame_score=slot['min_frame_score'],
                          x1=first[0], y1=first[1], x2=first[2], y2=first[3],
                          reason='DINO surface detection; foreground exclusion; continuous pixel/flow tracking; periodic detector rechecks.'))
        for row in rows:
            coordinates.append([slot['slot_id'], row[0], row[0]/fps, *row[1], row[2], row[3]])
    keys = ['slot_id','rank_order','track_id','scene_id','kind','start_sec','end_sec','start_time',
            'end_time','duration_sec','start_frame','end_frame_exclusive','ranking_score','min_frame_score',
            'x1','y1','x2','y2','reason']
    with (out_dir / 'slots.csv').open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(slots)
    with (out_dir / 'coordinates.csv').open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['slot_id','frame','time_sec','x1','y1','x2','y2','frame_rank','tracking_quality'])
        writer.writerows(coordinates)
    import importlib.metadata
    versions = {}
    for package in ['torch','torchvision','transformers','numpy','opencv-python-headless','Pillow']:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = 'not installed under this distribution name'
    digest = hashlib.sha256()
    with source.open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            digest.update(block)
    data = dict(pipeline_version='4.1',status='ok' if slots else 'no_suitable_slot', input_video=str(source.resolve()),
                input_sha256=digest.hexdigest(), config=asdict(cfg), metadata=meta, slots=slots,
                elapsed_sec=round(time.time()-started, 2), device=models.device,
                python=platform.python_version(), versions=versions,
                limitations=['Pretrained detector, not a custom-trained ad-suitability model.',
                             'Surface and foreground detections are sampled; flow/pixel tracking runs between them.',
                             'Axis-aligned boxes do not establish 3D planarity or perspective corners.',
                             'Ranking score is not calibrated confidence or measured accuracy.',
                             'No guarantee of detecting every suitable slot or excluding every unsuitable one.'])
    (out_dir / 'slots.json').write_text(json.dumps(data, indent=2), encoding='utf-8')
    return data


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--video', required=True)
    parser.add_argument('--out-dir', required=True)
    parser.add_argument('--max-slots', type=int, choices=[1,2,3], default=3)
    parser.add_argument('--min-duration', type=float, default=4.0)
    parser.add_argument('--target-duration', type=float, default=5.0)
    parser.add_argument('--detect-hz', type=float, default=1.0)
    parser.add_argument('--min-conf', type=float, default=.28)
    parser.add_argument('--min-rank', type=float, default=.46)
    parser.add_argument('--allow-existing', dest='allow_existing', action='store_true', default=True)
    parser.add_argument('--blank-only', dest='allow_existing', action='store_false')
    parser.add_argument('--model-id', default='IDEA-Research/grounding-dino-tiny')
    parser.add_argument('--cache', default=None, help='Optional raw detection cache; bound to source and analysis config.')
    parser.add_argument('--max-width', type=int, default=1280)
    parser.add_argument('--device', choices=['cpu','cuda'], default=None)
    args = parser.parse_args()
    if not 4 <= args.min_duration <= args.target_duration <= 5:
        parser.error('Require 4 <= min-duration <= target-duration <= 5 seconds.')
    if not 1 <= args.detect_hz <= 10:
        parser.error('detect-hz must be between 1 and 10.')
    if not .1 <= args.min_conf <= .9 or not 0 <= args.min_rank <= 1:
        parser.error('Invalid score thresholds.')
    if args.max_width < 320 or args.max_width % 2:
        parser.error('max-width must be an even integer >= 320.')
    return args


def main():
    args = parse_args()
    source = Path(args.video)
    if not source.is_file():
        raise FileNotFoundError(source)
    out_dir = Path(args.out_dir)
    # Refuse stale/mixed outputs. Choose a new per-video folder for every run.
    if out_dir.exists() and any(out_dir.iterdir()):
        raise FileExistsError(f'{out_dir} is not empty. Use a new run folder.')
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg = Config(**{key: getattr(args, key) for key in ['max_slots','min_duration','target_duration',
                  'detect_hz','min_conf','min_rank','allow_existing','model_id']})
    started = time.time()
    normalized = out_dir / 'normalized_input.mp4'
    normalize_video(source, normalized, args.max_width)
    digest = hashlib.sha256(source.read_bytes()).hexdigest()
    cache_name = f'{digest[:20]}_w{args.max_width}_hz{cfg.detect_hz:g}.json'
    cache_path = (Path(args.cache) / cache_name) if args.cache else (out_dir / 'raw_detections.json')
    models = Models(cfg, args.device, cache_path)
    tracks, meta = analyze(normalized, models, cfg)
    meta['fresh_model_inference_sec']=round(models.inference_seconds,3)
    meta['raw_detection_cache_entries']=len(models.cache)
    (out_dir/'raw_detections.json').write_text(json.dumps({'prompt':models.PROMPT,'model_id':cfg.model_id,
                                                         'source_sha256':digest,'frames':models.cache},indent=2),encoding='utf-8')
    preview_tracks, candidate_summaries = candidate_report(tracks,out_dir,meta['fps'],cfg)
    meta['detected_candidate_tracks'] = len(candidate_summaries)
    meta['longest_candidate_duration_sec'] = max((c['continuous_duration_sec'] for c in candidate_summaries),default=0)
    options = window_options(tracks, meta['fps'], cfg)
    selected = select_windows(options, cfg)
    meta['track_count'] = len(tracks)
    meta['valid_window_candidate_count'] = len(options)
    written = render(normalized, out_dir / 'phase3_suggestions.mp4', selected, meta['fps'])
    render(normalized, out_dir / 'detection_preview.mp4', preview_tracks, meta['fps'], preview=True)
    if written != meta['processed_frames']:
        raise RuntimeError('Analysis and rendering frame counts differ.')
    data = export_results(source, out_dir, cfg, selected, meta, started, models)
    print('\nStatus:', data['status'], '| Total slots:', len(data['slots']))
    print('Detected candidate tracks:',len(candidate_summaries),'| Longest continuous candidate:',meta['longest_candidate_duration_sec'],'s')
    if not data['slots']:
        print('Inspect detection_preview.mp4 and candidate_summary.csv: detections are separate from 4-second eligibility.')
    for slot in data['slots']:
        print(f"SLOT {slot['slot_id']}: {slot['start_time']} -> {slot['end_time']} ({slot['duration_sec']:.2f}s)")
    print('Saved:', out_dir.resolve())


if __name__ == '__main__':
    main()


Writing ad_slot_detector_v4.py


## 4. Upload clean original videos

For the first run choose one simple 6-10 second continuous blank-wall/empty-board
shot and one negative example. Uploaded videos can be processed together.
Do not upload the old annotated outputs as clean ground truth.


In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO_PATHS = [str(Path(name).resolve()) for name in uploaded
               if Path(name).suffix.lower() in {".mp4", ".mov", ".mkv", ".avi", ".webm"}]
assert VIDEO_PATHS, "No supported video was uploaded."
print("Selected originals:", VIDEO_PATHS)
del uploaded


Saving 20653317-uhd_2160_3840_30fps.mp4 to 20653317-uhd_2160_3840_30fps.mp4
Selected originals: ['/content/20653317-uhd_2160_3840_30fps.mp4']


## 5. Run V4 (existing boards ON)

Keep `ALLOW_EXISTING=True`. False means blank-wall-only and excludes an already
occupied billboard. Start with `DETECT_HZ=1`, matching the locally tested setting.
The 4-second minimum stays in force for FINAL slots, not for detection previews.
Each run gets a new output folder. A raw detection cache speeds repeat runs on
the exact same input/configuration; it never adds manually placed coordinates.


In [ ]:
from datetime import datetime
import re
ALLOW_EXISTING = True
DETECT_HZ = 1
MAX_SLOTS = 3
MIN_DURATION = 4.0
TARGET_DURATION = 5.0
RUN_ROOT = Path("phase3_results") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_ROOT.mkdir(parents=True, exist_ok=False)
RUN_DIRS = []
for index, source in enumerate(VIDEO_PATHS, 1):
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(source).stem)[:60]
    folder = RUN_ROOT / f"{index:02d}_{safe_name}"
    command = [sys.executable, "ad_slot_detector_v4.py", "--video", source,
               "--out-dir", str(folder), "--max-slots", str(MAX_SLOTS),
               "--min-duration", str(MIN_DURATION), "--target-duration", str(TARGET_DURATION),
               "--detect-hz", str(DETECT_HZ)]
    command.extend(["--cache", "phase3_raw_cache"])
    if not ALLOW_EXISTING:
        command.append("--blank-only")
    print("Processing:", source, "->", folder, flush=True)
    subprocess.run(command, check=True)
    RUN_DIRS.append(folder)
print("Completed result folders:", RUN_DIRS)


Processing: /content/20653317-uhd_2160_3840_30fps.mp4 -> phase3_results/20261116_153645_457391/01_20653317-uhd_2160_3840_30fps


## 6. Preview and inspect

Read every selected window in full. Score is a ranking index, not accuracy.
CSV coordinates are in normalized_input.mp4 pixels (max width 1280).
Use coordinates.csv for the moving per-frame location, not the first box only.


In [ ]:
from IPython.display import Video, display
import pandas as pd
for folder in RUN_DIRS:
    result = json.loads((folder / "slots.json").read_text())
    meta = result["metadata"]
    print(folder.name, "detected tracks:", meta["detected_candidate_tracks"],
          "longest:", meta["longest_candidate_duration_sec"], "seconds",
          "final slots:", len(result["slots"]))
    display(pd.read_csv(folder / "candidate_summary.csv"))
    print("DETECTION PREVIEW (candidates can be shorter than 4 seconds):")
    display(Video(str(folder / "detection_preview.mp4"), embed=True, width=960))
    display(pd.read_csv(folder / "slots.csv"))
    if result["slots"]:
        print("FINAL PHASE 3 WINDOWS (max 3; 4-5 seconds each):")
        display(Video(str(folder / "phase3_suggestions.mp4"), embed=True, width=960))
    elif meta["detected_candidate_tracks"]:
        print("Surface detections exist. Check candidate_summary.csv for their duration;")
        print("the 4-second eligibility/ranking rule prevented a final selection.")
    else:
        print("No candidate survived. Exact rejection counts:", meta["rejected_counts"])


01_20653317-uhd_2160_3840_30fps detected tracks: 0 longest: 0 seconds final slots: 0


,candidate_id,track_id,kind,start_sec,end_sec,continuous_duration_sec,detector_confirmed_span_sec,meets_4s_rule,reason


DETECTION PREVIEW (candidates can be shorter than 4 seconds):
Buffered data was truncated after reaching the output size limit.

## 7. Check output contracts

These checks validate count, duration, coordinates/frame counts and boundaries.
They do NOT measure whether a surface is actually a good place for an ad.
Human annotations and held-out evaluation are required for that.


In [ ]:
for folder in RUN_DIRS:
    data = json.loads((folder / "slots.json").read_text())
    slots = data["slots"]
    fps = data["metadata"]["fps"]
    assert len(slots) <= 3
    counts = {}
    with (folder / "coordinates.csv").open(newline="") as handle:
        for row in csv.DictReader(handle):
            sid = int(row["slot_id"])
            counts[sid] = counts.get(sid, 0) + 1
    for slot in slots:
        assert 4 - 1e-6 <= slot["duration_sec"] <= 5 + 1e-6
        expected = slot["end_frame_exclusive"] - slot["start_frame"]
        assert counts[slot["slot_id"]] == expected
        assert abs(expected / fps - slot["duration_sec"]) < 1e-6
        assert not any(slot["start_frame"] < cut < slot["end_frame_exclusive"]
                       for cut in data["metadata"]["scene_cut_frames"])
    for left, right in zip(slots, slots[1:]):
        assert left["end_frame_exclusive"] <= right["start_frame"]
    cap = cv2.VideoCapture(str(folder / "phase3_suggestions.mp4"))
    assert cap.isOpened(), "Cannot open rendered result"
    assert int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) == data["metadata"]["processed_frames"]
    cap.release()
    print(folder.name, ": output contracts PASS; visual/ML quality still needs review.")


01_17559124_1920_1080_30fps : output contracts PASS; visual/ML quality still needs review.


## 8. Download all results

The ZIP contains normalized source copies, final videos, timestamps, moving
coordinates, previews and metadata/logs. Download before disconnecting the runtime.


In [ ]:
archive = shutil.make_archive(str(RUN_ROOT) + "_all_results", "zip", root_dir=RUN_ROOT)
files.download(archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Before faculty demo

- Compare original V2 and V3 on the same clean inputs.
- Include one validated positive example and one correct no-slot example.
- Report actual usefulness precision, false positives/min, duration compliance,
  dataset size and measured runtime; no invented accuracy.
- Explain that pretrained models are used and surface/foreground detections are sampled; flow/pixel checks run
  on intervening frames. Generic semantic segmentation is not required in V4.
- Keep offline result videos, CSV/JSON and the report ready.
- Follow the guide for custom dataset/fine-tuning, perspective insertion,
  contextual ad selection and platform integration through the 8th semester.
